* Examine why large flexible networks can sometimes perform well on new data, and how we reduce overfitting risk in practice.

In [ ]:
import math
import random
import numpy as np
import torch

# 5.5.1 Revisiting Overfitting and Regularization

## 1. Intuition

* Overfitting means a model fits training data too specifically (e.g. memorizes it) and performs worse on new data.

* Regularization is any method that tries to improve generalization by constraining or guiding the learning process.

## 2. Why this exists

* Deep networks can have many parameters, so training loss alone is not enough evidence of useful learning.

* Decreasing training loss doesn't necessarily mean the model learned something that will work well on new data (test loss).

## 3. Examples

* Compare train and validation losses.

In [ ]:
train_loss = torch.tensor([1.0, 0.5, 0.2])
valid_loss = torch.tensor([1.1, 0.7, 0.9])
gap = valid_loss - train_loss # [1.1 - 1.0, 0.7 - 0.5, 0.9 - 0.2] = [0.1, 0.2, 0.7]
gap

tensor([0.1000, 0.2000, 0.7000])

## 4. Step-by-step breakdown

* Training loss decreases across epochs.

* Validation loss first decreases and then increases, as the model becomes increasingly susceptible to overfitting.

* A widening gap between training and validation loss can indicate overfitting.

* Regularization methods try to reduce this problem.

## 5. Connection to ML systems

Common regularization methods include:
* Weight decay
* Dropout
* Data augmentation
* Early stopping
* Model-size control

## 6. Common confusion points

- Low training loss is not the final goal.
- Regularization can increase training loss while improving validation performance.
- Overfitting is diagnosed with held-out (test) data.
- No regularization method is universally best.

# 5.5.2 Inspiration from Nonparametrics

## 1. Intuition

* Nonparametric methods like Decision Trees can grow in complexity with the amount of data.
> * In nonparametric models, model complexity can grow as more data is observed.
> * Whereas parametric models have a fixed-sized parameter structure determined by the model specification.

## 2. Why this exists

* This perspective helps explain why flexibility is not automatically bad

* Flexible models need enough data, suitable structure, and careful evaluation.

    | Model                         | Nonparametric? | Why                                                              |
    | ----------------------------- | -------------- | ---------------------------------------------------------------- |
    | **K-nearest neighbors (KNN)** | ✅              | Complexity/storage can grow with the dataset                     |
    | **Decision trees**            | ✅ Usually      | Tree structure can grow with the data                            |
    | **Random forests**            | ✅ Usually      | Collection of trees can represent increasingly complex structure |
    | **Kernel methods**            | ✅ Often        | Complexity can depend on the number of training examples         |
    | **Gaussian processes**        | ✅              | Function complexity is not fixed to a finite parameter vector    |
    | **Kernel density estimation** | ✅              | Density estimate becomes more detailed with more data            |
    | **Linear regression**         | ❌              | Fixed number of coefficients                                     |
    | **Logistic regression**       | ❌              | Fixed number of coefficients                                     |
    | **Neural networks**           | ❌              | Fixed parameter count for a given architecture                   |
    | **Transformers/LLMs**         | ❌              | Fixed parameter count for a given architecture                   |


## 3. Examples

* A tiny nearest-neighbor style prediction.

In [ ]:
train_x = torch.tensor([0.0, 0.1, 3.0])
train_y = torch.tensor([0.0, 2.0, 6.0])
query = torch.tensor(1.2)
dist = torch.abs(train_x - query) # abs([0.0 - 1.2, 0.1 - 1.2, 3.0 - 1.2]) = abs([-1.2, -1.1, -1.8]) = [1.2, 1.1, 1.8]
pred = train_y[torch.argmin(dist)] # dist[1] contains the smallest value, so train_y[1] = 2
pred

tensor(2.)

## 4. Step-by-step breakdown

* The query is compared with stored training inputs.

* `torch.abs(train_x - query)` calculates the absolute distance between the query and each stored input.

* `argmin` identifies the closest training input, and the prediction reuses the corresponding training output `(train_y)` from that nearest example.

* The query is positioned in the same space as the training inputs. We measure how close the query is to each training input, choose the closest one, and then use the output associated with that training input as the prediction.

## 5. Connection to ML systems

* Large neural networks can also use data-rich flexibility, though their mechanism differs from nearest neighbors.

## 6. Common confusion points

- Nonparametric does not mean simple.
- Memorization and useful flexibility can look similar without proper evaluation.
- More data can support more flexible models (e.g. CNN, Transformer).
- This section is conceptual motivation, not a training recipe.

# 5.5.3 Early Stopping

## 1. Intuition

* Early stopping means stopping training when **validation performance** stops improving.

* It treats the number of training epochs as a regularization choice.

## 2. Why this exists

* A model may begin to overfit if training continues after validation loss starts worsening.

## 3. Examples

* Select the epoch with the lowest validation loss.

In [5]:
valid_loss = torch.tensor([0.9, 0.6, 0.5, 0.7])
best_epoch = int(torch.argmin(valid_loss)) # 0.5 is the lowest element, so best_epoch = 2 for 0.5 is in index 2 of valid_loss
best_loss = valid_loss[best_epoch] # 0.5
best_epoch, best_loss

(2, tensor(0.5000))

## 4. Step-by-step breakdown

* The validation-loss tensor stores 1 value per epoch.

* `argmin` finds the position of the smallest validation loss.

* That position is the chosen stopping point in this toy example.

## 5. Connection to ML systems

* Training systems often save the best validation checkpoint instead of simply keeping the final epoch.
> * It can be as simple as a helper that tracks the lowest `val_loss` and, whenever a new best is found, saves a copy of the model's weights and biases (e.g., `if val_loss < best_val_loss: best_val_loss = val_loss; best_state = copy.deepcopy(model.state_dict())`).

## 6. Common confusion points

- Early stopping uses validation data, not test data.
- The best epoch is a hyperparameter chosen by validation performance.
- Noisy validation curves can make stopping decisions unstable.
> * A relatively small validation set
> * Randomness in the data
> * Stochastic training
> * The model's predictions changing from epoch to epoch
> * Measurement variation in the validation metric

- Checkpointing is needed if the best epoch is not the final epoch.

# 5.5.4 Classical Regularization Methods for Deep Networks

## 1. Intuition

* Classical regularization methods include weight decay, dropout, data augmentation, and model-size control.

* Data augmentation means creating modified training examples that preserve the label, such as cropping or flipping images.

## 2. Why this exists

* Each method limits overfitting in a different way:
> * ***Penalizing params***: Discourages overly complex parameter values (e.g., L2 regularization/weight decay)
> * ***Adding noise***: Makes training problem less predictable, encouraging the model to learn rather than memorize
> * ***Increasing data variety***: Exposes the model to more variations, making simple memorization less useful
> * ***Reducing model capacity***: Limits model expressiveness to avoid noise fitting (fewer layers, fewer params)

## 3. Examples

In [6]:
methods = {
    "weight_decay": "penalize large weights",
    "dropout": "randomly hide activations",
    "augmentation": "add label-preserving variants", # Like creating variants of a cat image from different positions and angles
    "smaller_model": "reduce capacity",
}

## 4. Step-by-step breakdown

* The dictionary maps method names to plain-English effects.

* Weight decay changes the objective.

* Dropout changes training-time activations.

* Augmentation the variety/diversity of the training data by adding label-preserving variants.

* Smaller models reduce the function class.

## 5. Connection to ML systems

* Real systems often combine several regularization methods and choose strengths using validation data.

## 6. Common confusion points

- Regularization is not a guarantee.
- Some methods are task-specific.
- Data augmentation must preserve labels.
- Validation performance should guide regularization strength.

# 5.5.5 Summary

## 1. Intuition

* Deep-learning generalization is managed through evaluation discipline and regularization methods.

* The practical goal is not minimum training loss, but reliable validation and test performance.

## 2. Why this exists

* Deep networks are flexible, so careful model selection and regularization are essential.

## 3. Examples

* A generalization checklist.

In [7]:
checks = [
    "track validation loss",
    "compare train/validation gap",
    "use regularization",
    "keep test set final",
]

## 4. Step-by-step breakdown

* The checklist follows the evaluation discipline.

* Training and validation curves diagnose behavior.

* Regularization methods are chosen using validation data.

* The test set remains final.

## 5. Connection to ML systems

* This mindset applies to every later deep model.

## 6. Common confusion points

- Generalization is measured on unseen data.
- Regularization can trade training fit for test/validation performance.
- Flexible models require careful evaluation.
- Keep records of hyperparameter choices.

# 5.5.6 Exercises

## 1. Intuition

* These exercises practice generalization diagnostics.

## 2. Why this exists

* The point is to read training curves as evidence, not decoration.

## 3. Examples

* Exercise 1: find the best validation epoch.

In [8]:
losses = torch.tensor([0.8, 0.55, 0.58, 0.7])
int(torch.argmin(losses)) # 0.55, or index 1

1

* Exercise 2: identify a growing train-validation gap.

In [9]:
train = torch.tensor([0.9, 0.4])
valid = torch.tensor([1.0, 0.9])
valid - train # [1 - 0.9, 0.9 - 0.4] or [0.1, 0.5]

tensor([0.1000, 0.5000])

## 4. Step-by-step breakdown

* Exercise 1 checks early stopping logic.

* Exercise 2 checks gap computation.

* A growing gap can signal overfitting.

## 5. Connection to ML systems

* These are simple versions of real experiment tracking.

## 6. Common confusion points

- Choose epochs using validation data.
- Do not tune repeatedly on the test set.
- A gap is evidence, not a full diagnosis.
- Curves should be interpreted with domain context.